In [ ]:
import os
import glob
import random
import torch
import numpy as np
import nibabel as nib
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

# --- 1. CONFIGURATION ---
CHECKPOINT_FOLDER = "checkpoints_attention_unet"
DATASET_ROOT = r"D:/Capstone/Experiment 4/Datasets/ISLES-2022" 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. ARCHITECTURE ---
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.InstanceNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.InstanceNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Conv2d(F_g, F_int, 1)
        self.W_x = nn.Conv2d(F_l, F_int, 1)
        self.psi = nn.Conv2d(F_int, 1, 1)
    def forward(self, g, x):
        psi = F.relu(self.W_g(g) + self.W_x(x))
        psi = torch.sigmoid(self.psi(psi))
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(2, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.dec3 = DoubleConv(256 + 128, 128)
        self.dec2 = DoubleConv(128 + 64, 64)
        self.dec1 = DoubleConv(64 + 32, 32)
        self.att3 = AttentionGate(256, 128, 64)
        self.att2 = AttentionGate(128, 64, 32)
        self.att1 = AttentionGate(64, 32, 16)
        self.out = nn.Conv2d(32, 2, 1)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        d3 = F.interpolate(e4, scale_factor=2, mode="bilinear", align_corners=False)
        e3 = self.att3(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = F.interpolate(d3, scale_factor=2, mode="bilinear", align_corners=False)
        e2 = self.att2(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = F.interpolate(d2, scale_factor=2, mode="bilinear", align_corners=False)
        e1 = self.att1(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)

# --- 3. DATA LOADER (Optimized) ---
class EvalDataset(Dataset):
    def __init__(self, root_dir, limit=50):
        self.items = []
        images_dir = os.path.join(root_dir, "imagesTr")
        labels_dir = os.path.join(root_dir, "labelsTr")
        
        # Get all files and shuffle them
        case_files = glob.glob(os.path.join(images_dir, "*_0000.nii.gz"))
        random.shuffle(case_files)
        
        # Only use 'limit' number of cases to speed things up
        selected_files = case_files[:limit]
        
        print(f"Indexing {len(selected_files)} random cases for rapid evaluation...")
        
        for f in selected_files:
            case_id = os.path.basename(f).replace("_0000.nii.gz", "")
            adc = f.replace("_0000.nii.gz", "_0001.nii.gz")
            label = os.path.join(labels_dir, f"{case_id}.nii.gz")
            
            if os.path.exists(adc) and os.path.exists(label):
                mask_vol = nib.load(label).get_fdata()
                depth = mask_vol.shape[2]
                for z in range(depth):
                    if mask_vol[:,:,z].max() > 0: # Only slices with stroke
                        self.items.append((f, adc, label, z))
                        
        print(f"Dataset Ready: {len(self.items)} positive slices.")

    def __len__(self): return len(self.items)
    
    def _norm(self, img): return (img - np.mean(img)) / (np.std(img) + 1e-8)

    def __getitem__(self, idx):
        dwi_p, adc_p, mask_p, z = self.items[idx]
        dwi = nib.load(dwi_p).get_fdata()[:, :, z]
        adc = nib.load(adc_p).get_fdata()[:, :, z]
        mask = nib.load(mask_p).get_fdata()[:, :, z]
        
        dwi_n = self._norm(dwi)
        adc_n = self._norm(adc)
        
        img = torch.tensor(np.stack([dwi_n, adc_n]), dtype=torch.float32)
        gt = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        
        img = F.interpolate(img.unsqueeze(0), size=(256, 256), mode='bilinear').squeeze(0)
        gt = F.interpolate(gt.unsqueeze(0), size=(256, 256), mode='nearest').squeeze(0)
        
        return img, gt

# --- 4. MAIN EVALUATION ---
checkpoints = glob.glob(os.path.join(CHECKPOINT_FOLDER, "*.pth"))
checkpoints.sort()

if not checkpoints:
    print("No checkpoints found!")
else:
    # Evaluate on 50 random patients (enough to find the best epoch)
    dataset = EvalDataset(DATASET_ROOT, limit=50) 
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

    best_dice = 0.0
    best_ckpt = ""
    
    print(f"\nComparing {len(checkpoints)} checkpoints...")
    print(f"{'Checkpoint Name':<40} | {'Mean Dice Score'}")
    print("-" * 60)

    for ckpt_path in tqdm(checkpoints):
        try:
            model = AttentionUNet().to(DEVICE)
            state = torch.load(ckpt_path, map_location=DEVICE)
            if 'model_state' in state: state = state['model_state']
            model.load_state_dict({k.replace("module.", ""): v for k, v in state.items()})
            model.eval()
            
            dice_scores = []
            with torch.no_grad():
                for x, y in dataloader:
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    logits = model(x)
                    
                    # --- FIX: ARGMAX FOR 2-CHANNEL OUTPUT ---
                    preds = torch.argmax(logits, dim=1).unsqueeze(1).float()
                    # ----------------------------------------
                    
                    intersection = (preds * y).sum()
                    union = preds.sum() + y.sum()
                    
                    if union == 0:
                        dice_scores.append(1.0)
                    else:
                        dice = (2. * intersection) / (union + 1e-6)
                        dice_scores.append(dice.item())
            
            avg_dice = np.mean(dice_scores)
            print(f"{os.path.basename(ckpt_path):<40} | {avg_dice:.5f}")
            
            if avg_dice > best_dice:
                best_dice = avg_dice
                best_ckpt = ckpt_path
                
        except Exception as e:
            print(f"Skipping {ckpt_path}: {e}")

    print("\n" + "="*30)
    print(f"🏆 ULTIMATE WINNER: {os.path.basename(best_ckpt)}")
    print(f"🏆 WITH DICE SCORE: {best_dice:.5f}")
    print("="*30)

Indexing 50 random cases for rapid evaluation...
Dataset Ready: 947 positive slices.

Comparing 101 checkpoints...
Checkpoint Name                          | Mean Dice Score
------------------------------------------------------------


  0%|          | 0/101 [00:00<?, ?it/s]C:\Users\ankit\AppData\Local\Temp\ipykernel_16244\1694408867.py:142: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(